# irAE scRNA-seq Pipeline
**Dataset:** GSE322576 (2026) — Inflammatory arthritis irAE, PBMC, CellRanger v3  
**Conditions:** irAE / HC (healthy control) / RAC (seroneg RA control) / ICI (checkpoint inhibitor, no irAE)  
**Project:** Ritschel Research / irae-scrna  
**Scripts:** 01 Load & QC → 02 scVI → 03 Annotate → 04 Signatures → 05 DE → 06 LINCS → 07 Novelty  

Run from the top after every runtime reset. Skip guards detect existing outputs automatically.  
**Key facts:** CellRanger v3 (features.tsv, 3 cols); `\w+` regex for flexible filename matching; `percent_top=None` in QC; `flavor='igraph'` in Leiden; scVI checkpoint saved immediately after training.

---
## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%capture
!pip install scanpy anndata scvi-tools gseapy igraph leidenalg

In [ ]:
import os, gc, re, time, requests
import numpy as np
import pandas as pd
import scipy.io
import anndata as ad
import scanpy as sc

sc.settings.verbosity = 1

DRIVE_BASE = '/content/drive/MyDrive/Ritschel_Research/irae_scrna_output'
RAW_DIR    = os.path.join(DRIVE_BASE, 'raw')
PROCESSED  = os.path.join(DRIVE_BASE, 'processed')

for d in [DRIVE_BASE, RAW_DIR, PROCESSED]:
    os.makedirs(d, exist_ok=True)

print('Drive base:', DRIVE_BASE)
print('Processed: ', PROCESSED)
print('Files in processed:', sorted(os.listdir(PROCESSED)))

---
## Script 01 — Load & QC
**CellRanger v3** — files named `{GSM}_{tag}_{barcodes|features|matrix}.tsv.gz/.mtx.gz`.  
features.tsv has 3 columns (Ensembl ID, gene symbol, feature type).  
Per-condition loading + concatenation for RAM management.  
**Fix:** `\w+` regex (not `\d+`) handles arbitrary patient tag strings.  
**Fix:** `percent_top=None` in `calculate_qc_metrics` avoids IndexError.  
**Note:** GSM9555177 (HC P11) has a corrupted download — skipped silently.  
**Output:** `processed/01_loaded.h5ad`

In [ ]:
_s01_out = os.path.join(PROCESSED, '01_loaded.h5ad')
if os.path.exists(_s01_out):
    print('01_loaded.h5ad found — skipping Script 01.')
else:
    print('Running Script 01...')

In [ ]:
CONDITION_MAP = {
    # irAE patients
    'GSM9555167': ('irAE', 'P1'),  'GSM9555180': ('irAE', 'P14'),
    'GSM9555181': ('irAE', 'P15'), 'GSM9555183': ('irAE', 'P17'),
    'GSM9555185': ('irAE', 'P19'), 'GSM9587734': ('irAE', 'P20'),
    'GSM9587747': ('irAE', 'P33'), 'GSM9587748': ('irAE', 'P34'),
    'GSM9587750': ('irAE', 'P36'), 'GSM9587752': ('irAE', 'P38'),
    # Healthy controls
    'GSM9555168': ('HC',   'P2'),  'GSM9555171': ('HC',   'P5'),
    'GSM9555174': ('HC',   'P8'),  'GSM9555175': ('HC',   'P9'),
    'GSM9555176': ('HC',   'P10'), 'GSM9555177': ('HC',   'P11'),  # corrupted — skip silently
    'GSM9587735': ('HC',   'P21'), 'GSM9587738': ('HC',   'P24'),
    'GSM9587741': ('HC',   'P27'), 'GSM9587742': ('HC',   'P28'),
    'GSM9587743': ('HC',   'P29'), 'GSM9587744': ('HC',   'P30'),
    # RA controls (seronegative)
    'GSM9555169': ('RAC',  'P3'),  'GSM9555170': ('RAC',  'P4'),
    'GSM9555172': ('RAC',  'P6'),  'GSM9555173': ('RAC',  'P7'),
    'GSM9555182': ('RAC',  'P16'), 'GSM9555184': ('RAC',  'P18'),
    'GSM9587736': ('RAC',  'P22'), 'GSM9587737': ('RAC',  'P23'),
    'GSM9587739': ('RAC',  'P25'), 'GSM9587740': ('RAC',  'P26'),
    'GSM9587749': ('RAC',  'P35'), 'GSM9587751': ('RAC',  'P37'),
    # ICI-treated (cancer, no irAE)
    'GSM9555178': ('ICI',  'P12'), 'GSM9555179': ('ICI',  'P13'),
    'GSM9587745': ('ICI',  'P31'), 'GSM9587746': ('ICI',  'P32'),
}

from collections import Counter
counts = Counter(v[0] for v in CONDITION_MAP.values())
print(f'Condition map: {dict(counts)}')

In [ ]:
MIN_GENES_PREFILTER = 200

def discover_samples(raw_dir):
    """CellRanger v3 flat layout.
    Uses \\w+ (not \\d+) to match any patient tag string.
    """
    samples = {}
    for fname in os.listdir(raw_dir):
        m = re.match(r'^(GSM\d+)_\w+_(barcodes|features|matrix)\.(tsv|mtx)\.gz$', fname)
        if not m: continue
        gsm, ftype = m.group(1), m.group(2)
        if gsm not in samples: samples[gsm] = {}
        samples[gsm][ftype] = os.path.join(raw_dir, fname)
    return samples


def load_sample_v3(gsm, paths, condition, patient):
    """CellRanger v3 loader.
    features.tsv: col 0 = Ensembl ID, col 1 = gene symbol, col 2 = feature type.
    Pre-filters empty droplets before constructing AnnData.
    """
    mat      = scipy.io.mmread(paths['matrix']).T.tocsr()
    barcodes = pd.read_csv(paths['barcodes'], header=None, sep='\t')[0].tolist()
    features = pd.read_csv(paths['features'], header=None, sep='\t')
    keep     = np.diff(mat.indptr) >= MIN_GENES_PREFILTER
    mat      = mat[keep, :]
    barcodes = [bc for bc, k in zip(barcodes, keep) if k]
    adata = ad.AnnData(X=mat)
    adata.obs_names            = [f'{gsm}_{bc}' for bc in barcodes]
    adata.var_names            = features[1].tolist()   # gene symbol
    adata.var['gene_ids']      = features[0].tolist()   # ensembl id
    adata.var['feature_types'] = features[2].tolist()   # 'Gene Expression'
    adata.obs['condition']     = condition
    adata.obs['patient']       = patient
    adata.obs['sample']        = gsm
    adata.var_names_make_unique()
    return adata


if not os.path.exists(_s01_out):
    print('Discovering samples...')
    all_samples = discover_samples(RAW_DIR)
    print(f'Found {len(all_samples)} GSM IDs')
    unknown = [g for g in all_samples if g not in CONDITION_MAP]
    if unknown: print(f'WARNING — no condition map for: {unknown} (skipping)')

    adatas_all = []
    for cond in ['irAE', 'HC', 'RAC', 'ICI']:
        print(f'\nLoading {cond} samples...')
        adatas = []
        for gsm in sorted(all_samples):
            if gsm not in CONDITION_MAP: continue
            c, patient = CONDITION_MAP[gsm]
            if c != cond: continue
            paths = all_samples[gsm]
            if not all(k in paths for k in ['barcodes','features','matrix']):
                print(f'  {gsm} — SKIP (missing files)'); continue
            try:
                a = load_sample_v3(gsm, paths, cond, patient)
                print(f'  {gsm} (patient {patient}) — {a.n_obs:,} cells')
                adatas.append(a)
            except Exception as e:
                print(f'  {gsm} — ERROR: {e}')
        if not adatas:
            print(f'  WARNING: no {cond} samples loaded'); continue
        print(f'  Concatenating {len(adatas)} {cond} samples...')
        batch = sc.concat(adatas, join='outer', fill_value=0)
        del adatas; gc.collect()
        print(f'  {cond}: {batch.n_obs:,} cells')
        adatas_all.append(batch)

    print('\nMerging all conditions...')
    adata = sc.concat(adatas_all, join='outer', fill_value=0)
    del adatas_all; gc.collect()
    print(f'Combined: {adata.n_obs:,} cells x {adata.n_vars:,} genes')
    print(adata.obs['condition'].value_counts().to_string())

In [ ]:
if not os.path.exists(_s01_out):
    # percent_top=None avoids IndexError when n_genes < 500 (not an issue with real ~30k gene data)
    adata.var['mt'] = adata.var_names.str.startswith('MT-')
    sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, inplace=True)
    print(f'Median genes/cell: {adata.obs["n_genes_by_counts"].median():.0f}')
    print(f'Median MT%:        {adata.obs["pct_counts_mt"].median():.1f}%')
    sc.pl.violin(adata, ['n_genes_by_counts','total_counts','pct_counts_mt'],
                 jitter=0.4, multi_panel=True)

In [ ]:
if not os.path.exists(_s01_out):
    # Adjust thresholds after inspecting violin plots above
    MIN_GENES  = 200
    MAX_GENES  = 6000
    MAX_MT_PCT = 20

    n_before = adata.n_obs
    adata = adata[adata.obs['n_genes_by_counts'] >= MIN_GENES].copy()
    adata = adata[adata.obs['n_genes_by_counts'] <= MAX_GENES].copy()
    adata = adata[adata.obs['pct_counts_mt']     <= MAX_MT_PCT].copy()
    print(f'After QC: {adata.n_obs:,} cells ({n_before - adata.n_obs:,} removed)')

    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    adata.raw = adata.copy()
    sc.pp.highly_variable_genes(adata, n_top_genes=3000, batch_key='sample', subset=True)
    print(f'HVGs: {adata.n_vars}')

    adata.write_h5ad(_s01_out)
    print(f'Saved: {_s01_out}')
    print('Script 01 complete.')

---
## Script 02 — scVI Embedding & Leiden Clustering
200 epochs with early stopping. **Checkpoint saved immediately after training** — safe to resume after runtime reset without retraining.  
**Fix:** `flavor='igraph', n_iterations=2, directed=False` in `sc.tl.leiden` — required for current scanpy/leidenalg versions.  
**Requires T4 GPU.**  
**Output:** `processed/02_scvi.h5ad`

In [ ]:
_s02_out  = os.path.join(PROCESSED, '02_scvi.h5ad')
_s02_ckpt = os.path.join(PROCESSED, '02_scvi_ckpt.h5ad')
if os.path.exists(_s02_out):
    print('02_scvi.h5ad found — skipping Script 02.')
elif os.path.exists(_s02_ckpt):
    print('Training checkpoint found — will resume from post-training state.')
else:
    print('Running Script 02 from scratch...')

In [ ]:
import scvi, torch, random

SCVI_PARAMS        = {'n_latent': 30, 'n_layers': 2, 'n_hidden': 128}
SCVI_TRAIN_PARAMS  = {'max_epochs': 200, 'early_stopping': True, 'early_stopping_patience': 20}
N_NEIGHBORS        = 15
RANDOM_SEED        = 0
LEIDEN_RESOLUTIONS = [0.5, 0.8, 1.2]

# PBMC cell type markers (9 types)
CELL_TYPE_MARKERS = {
    'cd4_t_cell':   ['CD3D','CD3E','CD4','IL7R','TRAC'],
    'cd8_t_cell':   ['CD3D','CD3E','CD8A','CD8B','GZMK','GZMB'],
    'treg':         ['FOXP3','CTLA4','IL7R','CD4','IKZF2'],
    'exhausted_t':  ['PDCD1','LAG3','TIGIT','HAVCR2','TOX','ENTPD1'],
    'nk_cell':      ['GNLY','NKG7','NCAM1','KLRD1','KLRB1','GZMB'],
    'b_cell':       ['CD79A','MS4A1','CD19','IGHM','IGHD','CD27'],
    'plasma_cell':  ['MZB1','JCHAIN','IGHG1','IGHA1','SDC1'],
    'monocyte':     ['CD14','LYZ','S100A8','S100A9','CCL2','FCGR3A'],
    'dc':           ['HLA-DRA','CLEC9A','FCER1A','CLEC10A','IRF8'],
}

np.random.seed(RANDOM_SEED); random.seed(RANDOM_SEED); torch.manual_seed(RANDOM_SEED)

In [ ]:
if not os.path.exists(_s02_out):
    if os.path.exists(_s02_ckpt):
        # Resume from post-training checkpoint — skip retraining
        print('Loading from checkpoint (training already done)...')
        adata = sc.read_h5ad(_s02_ckpt)
        print(f'  {adata.n_obs:,} cells x {adata.n_vars} HVGs')
        print(f'  X_scVI present: {"X_scVI" in adata.obsm}')
    else:
        # Full training path
        print('Loading 01_loaded.h5ad...')
        adata = sc.read_h5ad(_s01_out)
        print(f'  {adata.n_obs:,} cells x {adata.n_vars} HVGs')

        scvi.settings.seed = RANDOM_SEED
        scvi.model.SCVI.setup_anndata(adata, batch_key='sample')
        model = scvi.model.SCVI(adata, **SCVI_PARAMS)
        print(f'Training scVI on {adata.n_obs:,} cells x {adata.n_vars} HVGs...')
        model.train(**SCVI_TRAIN_PARAMS, accelerator='auto')
        try:
            key = 'train_loss_epoch' if 'train_loss_epoch' in model.history else 'train_loss'
            print(f'Training complete. Final loss: {float(np.array(model.history[key].values[-1]).flat[0]):.2f}')
        except Exception:
            print('Training complete.')

        adata.obsm['X_scVI'] = model.get_latent_representation()
        print(f'Latent shape: {adata.obsm["X_scVI"].shape}')

        # Save checkpoint immediately — protects against timeout before UMAP/Leiden
        adata.write_h5ad(_s02_ckpt)
        print(f'Checkpoint saved: {_s02_ckpt}')

In [ ]:
if not os.path.exists(_s02_out):
    print('Building neighbour graph and UMAP...')
    sc.pp.neighbors(adata, use_rep='X_scVI', n_neighbors=N_NEIGHBORS)
    sc.tl.umap(adata)
    print('UMAP complete')

In [ ]:
if not os.path.exists(_s02_out):
    res_results = {}
    for res in LEIDEN_RESOLUTIONS:
        key = f'leiden_{res}'
        # flavor='igraph' required for current scanpy/leidenalg versions
        sc.tl.leiden(adata, resolution=res, key_added=key,
                     random_state=RANDOM_SEED, flavor='igraph',
                     n_iterations=2, directed=False)
        n = adata.obs[key].nunique()
        print(f'  Resolution {res}: {n} clusters')
        res_results[res] = {'key': key, 'n_clusters': n}

    # Score resolutions by cell type separation
    rows = []
    for res, info in res_results.items():
        best_clusters = {}
        for ct, markers in CELL_TYPE_MARKERS.items():
            present = [m for m in markers if m in adata.var_names]
            if not present: continue
            cluster_means = {}
            for cl in adata.obs[info['key']].unique():
                mask = adata.obs[info['key']] == cl
                e = adata[mask, present].X
                if hasattr(e, 'toarray'): e = e.toarray()
                cluster_means[cl] = float(e.mean())
            best_clusters[ct] = max(cluster_means, key=cluster_means.get)
        n_distinct = len(set(best_clusters.values()))
        n_resolved = len(best_clusters)
        rows.append({'resolution': res, 'n_clusters': info['n_clusters'],
                     'n_celltypes_resolved': n_resolved,
                     'n_distinct_best_clusters': n_distinct,
                     'separation_score': n_distinct / max(n_resolved, 1)})

    res_df = pd.DataFrame(rows).sort_values('separation_score', ascending=False)
    print('\nResolution comparison:')
    print(res_df.to_string(index=False))
    recommended = float(res_df.iloc[0]['resolution'])
    print(f'\nRecommended: {recommended} ({int(res_df.iloc[0]["n_clusters"])} clusters)')

    adata.obs['leiden'] = adata.obs[f'leiden_{recommended}'].copy()
    adata.uns['recommended_leiden_resolution'] = recommended
    adata.uns['pro_irae_clusters'] = []

    adata.write_h5ad(_s02_out)
    res_df.to_csv(os.path.join(PROCESSED, 'resolution_metrics.csv'), index=False)
    adata.obs.groupby(['leiden','condition']).size().unstack(fill_value=0).to_csv(
        os.path.join(PROCESSED, 'cluster_condition_distribution.csv'))
    print(f'Saved: {_s02_out}')
    print(f'Clusters: {adata.obs["leiden"].nunique()}')
    print('Script 02 complete.')

---
## Script 03 — Cell Type Annotation
9 PBMC cell types scored per cluster using marker gene expression.  
**Output:** `processed/03_annotated.h5ad`

In [ ]:
_s03_out = os.path.join(PROCESSED, '03_annotated.h5ad')
if os.path.exists(_s03_out):
    print('03_annotated.h5ad found — skipping Script 03.')
else:
    CONFIDENCE_THRESHOLD = 1.2

    print('Loading 02_scvi.h5ad...')
    adata = sc.read_h5ad(_s02_out)
    print(f'  {adata.n_obs:,} cells, {adata.obs["leiden"].nunique()} clusters')
    if 'norm_log' not in adata.layers:
        sc.pp.normalize_total(adata, target_sum=1e4); sc.pp.log1p(adata)

    clusters = sorted(adata.obs['leiden'].unique(), key=lambda x: int(x))
    scores = {ct: [] for ct in CELL_TYPE_MARKERS}
    for cl in clusters:
        mask = adata.obs['leiden'] == cl
        for ct, markers in CELL_TYPE_MARKERS.items():
            present = [m for m in markers if m in adata.var_names]
            if not present: scores[ct].append(0.0); continue
            e = adata[mask, present].X
            if hasattr(e, 'toarray'): e = e.toarray()
            scores[ct].append(float(e.mean()))

    score_df = pd.DataFrame(scores, index=clusters)
    score_df.index.name = 'cluster'

    ann_rows = []
    for cluster in score_df.index:
        row = score_df.loc[cluster]
        best = row.idxmax(); best_score = row.max()
        runner_up = row.sort_values(ascending=False).iloc[1] if len(row) > 1 else 0.0
        if best_score == 0.0:
            label, conf = 'unknown', 'low'
        elif runner_up == 0.0 or best_score >= CONFIDENCE_THRESHOLD * runner_up:
            label, conf = best, 'high'
        else:
            label, conf = best + '_mixed', 'low'
        ann_rows.append({'cluster': cluster, 'annotation': label, 'confidence': conf,
                         'best_score': round(best_score, 4),
                         'runner_up_score': round(runner_up, 4)})

    ann_df = pd.DataFrame(ann_rows)
    counts  = adata.obs['leiden'].value_counts().rename('n_cells')
    summary = ann_df.set_index('cluster').join(counts).sort_values('n_cells', ascending=False)
    print('\nCluster annotations:')
    print('-' * 75)
    for cl, row in summary.iterrows():
        print(f'  Cluster {cl:>3} | {row["annotation"]:<25} | {row["confidence"]:<4} | {int(row["n_cells"]):>6} cells')

    adata.obs['cell_type'] = adata.obs['leiden'].map(
        dict(zip(ann_df['cluster'].astype(str), ann_df['annotation'])))
    adata.obs['annotation_confidence'] = adata.obs['leiden'].map(
        dict(zip(ann_df['cluster'].astype(str), ann_df['confidence'])))

    adata.write_h5ad(_s03_out)
    ann_df.to_csv(os.path.join(PROCESSED, 'cluster_annotations.csv'), index=False)
    score_df.to_csv(os.path.join(PROCESSED, 'cluster_marker_scores.csv'))
    print(f'\nSaved: {_s03_out}')
    print('Script 03 complete.')

---
## Script 04 — irAE Signature Scoring
5 irAE-relevant gene signatures scored across all 4 conditions.  
**Output:** `processed/04_scored.h5ad`

In [ ]:
_s04_out = os.path.join(PROCESSED, '04_scored.h5ad')
if os.path.exists(_s04_out):
    print('04_scored.h5ad found — skipping Script 04.')
else:
    IRAE_SIGNATURES = {
        't_cell_exhaustion_activation': [
            'PDCD1','LAG3','TIGIT','HAVCR2','TOX','GZMK','GZMB','IFNG',
            'PRF1','NKG7','ENTPD1','CTLA4','CD38','CXCR3'],
        'th1_th17_inflammation': [
            'IFNG','IL17A','CXCL10','CXCL9','STAT1','IRF1','TBX21',
            'RORC','IL6','IL18','IL23A','IL12A','CCL5','CXCL13'],
        'checkpoint_dysregulation': [
            'PDCD1','CTLA4','CD274','PDCD1LG2','CD80','CD86',
            'ICOS','ICOSLG','LAG3','TIGIT','HAVCR2','VSIR'],
        'cytokine_storm': [
            'IL1B','TNF','CXCL8','IL6','CCL2','CCL5','CXCL1',
            'IL18','IL23A','IL12A','CXCL9','CXCL10','IL1A'],
        'isg_interferon_response': [
            'ISG15','MX1','OAS1','IFIT1','IFIT3','IFI44L',
            'STAT1','IRF1','CXCL10','CXCL9','IFI6','RSAD2','HERC5'],
    }

    print('Loading 03_annotated.h5ad...')
    adata = sc.read_h5ad(_s03_out)
    print(f'  {adata.n_obs:,} cells x {adata.n_vars} genes')
    if 'norm_log' not in adata.layers:
        sc.pp.normalize_total(adata, target_sum=1e4); sc.pp.log1p(adata)

    clusters = sorted(adata.obs['leiden'].unique(), key=lambda x: int(x))
    cluster_scores = {sig: [] for sig in IRAE_SIGNATURES}
    print('Scoring irAE signatures:')
    for sig_name, genes in IRAE_SIGNATURES.items():
        present = [g for g in genes if g in adata.var_names]
        missing = [g for g in genes if g not in adata.var_names]
        msg = f'  {sig_name}: {len(present)}/{len(genes)} genes'
        if missing: msg += f' (missing: {missing[:3]}{"..." if len(missing)>3 else ""})'
        print(msg)
        if not present:
            adata.obs['score_'+sig_name] = 0.0
            cluster_scores[sig_name].extend([0.0]*len(clusters)); continue
        e = adata[:, present].X
        if hasattr(e, 'toarray'): e = e.toarray()
        cell_scores = np.array(e.mean(axis=1)).flatten()
        adata.obs['score_'+sig_name] = cell_scores
        for cl in clusters:
            mask = adata.obs['leiden'] == cl
            cluster_scores[sig_name].append(float(cell_scores[mask].mean()))

    sig_df = pd.DataFrame(cluster_scores, index=clusters)
    sig_df.index.name = 'cluster'
    sig_df['irae_primary_score'] = (sig_df.get('t_cell_exhaustion_activation', 0) +
                                    sig_df.get('th1_th17_inflammation', 0))
    sig_df['irae_composite_score'] = sig_df[[c for c in IRAE_SIGNATURES
                                             if c in sig_df.columns]].sum(axis=1)

    top_clusters = sig_df.nlargest(3, 'irae_primary_score')
    print('\nTop 3 pro-irAE clusters:')
    for cl, row in top_clusters.iterrows():
        ct = adata.obs.loc[adata.obs['leiden']==str(cl), 'cell_type'].values
        ct = ct[0] if len(ct) > 0 else 'unknown'
        print(f'  Cluster {cl} ({ct}): exhaustion/act={row.get("t_cell_exhaustion_activation",0):.4f}, '
              f'th1/17={row.get("th1_th17_inflammation",0):.4f}')

    # Scores by condition — all 4 groups
    cond_rows = []
    for cond in ['irAE', 'HC', 'RAC', 'ICI']:
        mask = adata.obs['condition'] == cond
        if not mask.any(): continue
        row = {'condition': cond, 'n_cells': int(mask.sum())}
        for sig in IRAE_SIGNATURES:
            col = 'score_'+sig
            row[sig] = round(float(adata.obs.loc[mask, col].mean()), 4) \
                       if col in adata.obs.columns else 0.0
        cond_rows.append(row)
    cond_df = pd.DataFrame(cond_rows)
    print('\nScores by condition:')
    print(cond_df[['condition','n_cells','t_cell_exhaustion_activation',
                   'th1_th17_inflammation','isg_interferon_response']].round(4).to_string(index=False))

    adata.uns['pro_irae_clusters'] = top_clusters.index.tolist()
    adata.write_h5ad(_s04_out)
    sig_df.to_csv(os.path.join(PROCESSED, 'signature_scores.csv'))
    cond_df.to_csv(os.path.join(PROCESSED, 'signature_scores_by_condition.csv'), index=False)
    print(f'\nSaved: {_s04_out}')
    print('Script 04 complete.')

---
## Script 05 — Differential Expression
**Primary:** irAE vs HC  
**Novelty filter:** irAE vs RAC (removes RA-shared signal)  
**Artifact filter:** irAE vs ICI (removes ICI-shared signal)  
Plus cluster-vs-rest, pro-irAE cluster, T cell subset.  
**Output:** `processed/05_de.h5ad` + CSVs

In [ ]:
_s05_out = os.path.join(PROCESSED, '05_de.h5ad')
if os.path.exists(_s05_out):
    print('05_de.h5ad found — skipping Script 05.')
else:
    N_TOP_GENES_DE = 150

    print('Loading 04_scored.h5ad...')
    adata = sc.read_h5ad(_s04_out)
    print(f'  {adata.n_obs:,} cells, {adata.obs["leiden"].nunique()} clusters')
    for cond in ['irAE','HC','RAC','ICI']:
        print(f'  {cond}: {(adata.obs["condition"]==cond).sum():,}')
    if 'norm_log' not in adata.layers:
        sc.pp.normalize_total(adata, target_sum=1e4); sc.pp.log1p(adata)

    # Primary: irAE vs HC
    print('\nirAE vs HC DE (primary)...')
    sc.tl.rank_genes_groups(adata, groupby='condition', groups=['irAE'],
                            reference='HC', method='wilcoxon',
                            use_raw=False, key_added='rank_genes_irae_vs_hc', pts=True)
    r = adata.uns['rank_genes_irae_vs_hc']
    irae_vs_hc = pd.DataFrame({
        'gene': r['names']['irAE'], 'score': r['scores']['irAE'],
        'pval_adj': r['pvals_adj']['irAE'], 'log2fc': r['logfoldchanges']['irAE'],
    }).sort_values('score', ascending=False)
    irae_vs_hc.to_csv(os.path.join(PROCESSED, 'de_irAE_vs_HC.csv'), index=False)
    print(f'  Top up:   {irae_vs_hc.head(5)["gene"].tolist()}')
    print(f'  Top down: {irae_vs_hc.tail(5)["gene"].tolist()}')

    # Secondary: irAE vs RAC (novelty filter)
    print('\nirAE vs RAC DE (novelty filter)...')
    sc.tl.rank_genes_groups(adata, groupby='condition', groups=['irAE'],
                            reference='RAC', method='wilcoxon',
                            use_raw=False, key_added='rank_genes_irae_vs_rac', pts=True)
    r2 = adata.uns['rank_genes_irae_vs_rac']
    irae_vs_rac = pd.DataFrame({
        'gene': r2['names']['irAE'], 'score': r2['scores']['irAE'],
        'pval_adj': r2['pvals_adj']['irAE'], 'log2fc': r2['logfoldchanges']['irAE'],
    }).sort_values('score', ascending=False)
    irae_vs_rac.to_csv(os.path.join(PROCESSED, 'de_irAE_vs_RAC.csv'), index=False)
    print(f'  Top up: {irae_vs_rac.head(5)["gene"].tolist()}')

    # Secondary: irAE vs ICI (artifact filter)
    print('\nirAE vs ICI DE (artifact filter)...')
    sc.tl.rank_genes_groups(adata, groupby='condition', groups=['irAE'],
                            reference='ICI', method='wilcoxon',
                            use_raw=False, key_added='rank_genes_irae_vs_ici', pts=True)
    r3 = adata.uns['rank_genes_irae_vs_ici']
    irae_vs_ici = pd.DataFrame({
        'gene': r3['names']['irAE'], 'score': r3['scores']['irAE'],
        'pval_adj': r3['pvals_adj']['irAE'], 'log2fc': r3['logfoldchanges']['irAE'],
    }).sort_values('score', ascending=False)
    irae_vs_ici.to_csv(os.path.join(PROCESSED, 'de_irAE_vs_ICI.csv'), index=False)
    print(f'  Top up: {irae_vs_ici.head(5)["gene"].tolist()}')

    # Cluster-vs-rest
    print('\nCluster-vs-rest DE...')
    sc.tl.rank_genes_groups(adata, groupby='leiden', method='wilcoxon',
                            use_raw=False, key_added='rank_genes_groups', pts=True)
    r4 = adata.uns['rank_genes_groups']
    rows_de = []
    for cl in r4['names'].dtype.names:
        df_cl = pd.DataFrame({
            'cluster': cl, 'gene': r4['names'][cl],
            'score': r4['scores'][cl], 'pval_adj': r4['pvals_adj'][cl]})
        rows_de.extend([df_cl.nlargest(N_TOP_GENES_DE, 'score').assign(direction='up'),
                        df_cl.nsmallest(N_TOP_GENES_DE, 'score').assign(direction='down')])
    top_genes_df = pd.concat(rows_de, ignore_index=True)
    top_genes_df.to_csv(os.path.join(PROCESSED, 'de_top_genes.csv'), index=False)
    print(f'  {len(top_genes_df):,} gene-cluster pairs')

    # Pro-irAE cluster DE
    pro_clusters = [str(c) for c in list(adata.uns.get('pro_irae_clusters', []))]
    if pro_clusters:
        adata.obs['pro_irae_group'] = adata.obs['leiden'].apply(
            lambda x: 'pro_irae' if str(x) in pro_clusters else 'other')
        sc.tl.rank_genes_groups(adata, groupby='pro_irae_group', groups=['pro_irae'],
                                reference='other', method='wilcoxon',
                                use_raw=False, key_added='rank_genes_proirae')
        pr = adata.uns['rank_genes_proirae']
        pd.DataFrame({'gene': pr['names']['pro_irae'], 'score': pr['scores']['pro_irae'],
                      'pval_adj': pr['pvals_adj']['pro_irae']}
                    ).sort_values('score', ascending=False
                    ).to_csv(os.path.join(PROCESSED, 'de_proirae_vs_rest.csv'), index=False)
        print('  Pro-irAE cluster DE saved.')

    # T cell subset DE
    if 'cell_type' in adata.obs.columns:
        tcell_mask = adata.obs['cell_type'].str.contains('t_cell|exhausted|treg', na=False)
        adata_tcell = adata[tcell_mask].copy()
        print(f'\nT cell subset: {adata_tcell.n_obs:,} cells')
        if adata_tcell.n_obs > 100 and (adata_tcell.obs['condition']=='irAE').sum() > 50:
            sc.tl.rank_genes_groups(adata_tcell, groupby='condition', groups=['irAE'],
                                    reference='HC', method='wilcoxon',
                                    use_raw=False, key_added='rank_tcell_irae')
            tr = adata_tcell.uns['rank_tcell_irae']
            pd.DataFrame({'gene': tr['names']['irAE'], 'score': tr['scores']['irAE'],
                          'pval_adj': tr['pvals_adj']['irAE']}
                        ).sort_values('score', ascending=False
                        ).to_csv(os.path.join(PROCESSED, 'de_tcell_irAE_vs_HC.csv'), index=False)
            print('  T cell DE saved.')
        else:
            print('  Insufficient T cells — skipping focused DE.')

    adata.write_h5ad(_s05_out)
    print(f'\nSaved: {_s05_out}')
    print('Script 05 complete.')

---
## Script 06 — LINCS L1000 Reversal Scoring
Queries: irAE vs HC (primary), irAE vs RAC, irAE vs ICI, pro-irAE clusters, T cell subset, per-cluster.  
**Output:** `processed/lincs_candidates.csv`

In [ ]:
_s06_out = os.path.join(PROCESSED, 'lincs_candidates.csv')
if os.path.exists(_s06_out):
    print('lincs_candidates.csv found — skipping Script 06.')
else:
    import gseapy as gp

    N_TOP_GENES_LINCS = 150
    TOP_PER_QUERY     = 15
    ENRICHR_DELAY     = 1.0
    ENRICHR_LIBRARIES = [
        'LINCS_L1000_Chem_Pert_up', 'LINCS_L1000_Chem_Pert_down',
        'GO_Biological_Process_2023', 'Reactome_2022', 'KEGG_2021_Human',
    ]

    def clean_compound_name(term):
        m = re.match(r'^LJP\d+\s+\S+\s+\S+?-(.+)-[\d.]+$', term.strip())
        if m: return m.group(1).strip()
        return term.split('_')[0].strip() if term else term.strip()

    def run_enrichr(query_id, up_genes, down_genes):
        results = []
        for direction, genes, reversal_lib in [
            ('up',   up_genes,   'LINCS_L1000_Chem_Pert_down'),
            ('down', down_genes, 'LINCS_L1000_Chem_Pert_up'),
        ]:
            if not genes: continue
            for lib in ENRICHR_LIBRARIES:
                try:
                    enr = gp.enrichr(gene_list=genes, gene_sets=lib, outdir=None, verbose=False)
                    df = enr.results.copy()
                    if df.empty: continue
                    df['query_id'] = query_id
                    df['query_direction'] = direction
                    df['library'] = lib
                    if lib in ('LINCS_L1000_Chem_Pert_up','LINCS_L1000_Chem_Pert_down'):
                        adj_p = df['Adjusted P-value'].clip(lower=1e-300)
                        sign  = 1.0 if lib == reversal_lib else -1.0
                        df['reversal_score'] = sign * (-np.log10(adj_p))
                        df['compound'] = df['Term'].apply(clean_compound_name)
                    else:
                        df['reversal_score'] = 0.0
                        df['compound'] = df['Term']
                    results.append(df)
                    time.sleep(ENRICHR_DELAY)
                except Exception as e:
                    print(f'    WARNING: [{query_id} {lib}]: {e}')
                    time.sleep(ENRICHR_DELAY * 2)
        return pd.concat(results, ignore_index=True) if results else pd.DataFrame()

    print('Loading DE gene lists...')
    irae_vs_hc   = pd.read_csv(os.path.join(PROCESSED, 'de_irAE_vs_HC.csv'))
    top_genes_df = pd.read_csv(os.path.join(PROCESSED, 'de_top_genes.csv'))
    print(f'  {len(irae_vs_hc):,} genes | {top_genes_df["cluster"].nunique()} clusters')

    all_results = []

    # Primary
    print('Primary query: irAE vs HC...')
    res = run_enrichr('irAE_vs_HC',
                      irae_vs_hc[irae_vs_hc['score']>0].head(N_TOP_GENES_LINCS)['gene'].tolist(),
                      irae_vs_hc[irae_vs_hc['score']<0].tail(N_TOP_GENES_LINCS)['gene'].tolist())
    if not res.empty:
        all_results.append(res)
        print(f'  {(res["reversal_score"]>0).sum()} reversal hits')

    # Secondary queries
    for fname, qid in [('de_irAE_vs_RAC.csv','irAE_vs_RAC'),
                       ('de_irAE_vs_ICI.csv','irAE_vs_ICI'),
                       ('de_proirae_vs_rest.csv','ProirAE_clusters'),
                       ('de_tcell_irAE_vs_HC.csv','Tcell_irAE_vs_HC')]:
        fpath = os.path.join(PROCESSED, fname)
        if os.path.exists(fpath):
            de = pd.read_csv(fpath)
            res = run_enrichr(qid,
                              de[de['score']>0].head(N_TOP_GENES_LINCS)['gene'].tolist(),
                              de[de['score']<0].tail(N_TOP_GENES_LINCS)['gene'].tolist())
            if not res.empty:
                all_results.append(res)
                print(f'  {qid}: {(res["reversal_score"]>0).sum()} hits')

    # Per-cluster
    n_cl = top_genes_df['cluster'].nunique()
    for i, cl in enumerate(top_genes_df['cluster'].unique()):
        print(f'  Cluster {cl} ({i+1}/{n_cl})...', end=' ', flush=True)
        up   = top_genes_df.loc[(top_genes_df['cluster']==cl) &
                                 (top_genes_df['direction']=='up'), 'gene'].tolist()
        down = top_genes_df.loc[(top_genes_df['cluster']==cl) &
                                 (top_genes_df['direction']=='down'), 'gene'].tolist()
        res  = run_enrichr(f'cluster_{cl}', up, down)
        if not res.empty:
            all_results.append(res)
            print(f'{(res["reversal_score"]>0).sum()} hits')
        else:
            print('no results')

    if not all_results:
        raise RuntimeError('No Enrichr results returned.')

    raw_results = pd.concat(all_results, ignore_index=True)
    raw_results.to_csv(os.path.join(PROCESSED, 'lincs_results_raw.csv'), index=False)
    print(f'\nRaw results: {len(raw_results):,} rows')

    lincs_df = raw_results[
        raw_results['library'].str.startswith('LINCS_L1000') &
        (raw_results['reversal_score'] > 0)].copy()
    top = lincs_df.sort_values('reversal_score', ascending=False
                    ).groupby('query_id').head(TOP_PER_QUERY)
    candidates = top.groupby('compound').agg(
        max_reversal_score=('reversal_score','max'),
        n_queries=('query_id','nunique'),
        queries=('query_id', lambda x: ','.join(sorted(set(x.astype(str))))),
    ).reset_index().sort_values('max_reversal_score', ascending=False)

    print(f'Unique compounds: {len(candidates)}')
    print(candidates[['compound','max_reversal_score','n_queries']].head(15).round(2).to_string(index=False))
    candidates.to_csv(_s06_out, index=False)
    print('Script 06 complete.')

---
## Script 07 — Novelty Assessment & Priority Scoring
irAE-specific PubMed queries: hits_irae, hits_immune_checkpoint, hits_arthritis.  
**Output:** `processed/priority_candidates.csv`, `processed/patent_watch.csv`

In [ ]:
_s07_out = os.path.join(PROCESSED, 'priority_candidates.csv')
if os.path.exists(_s07_out):
    print('priority_candidates.csv found — skipping Script 07.')
else:
    NCBI_ESEARCH = 'https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi'
    NCBI_DELAY   = 0.4
    NCBI_EMAIL   = 'glen.ritschel@ritschelresearch.com'
    NOVELTY_WEIGHTS = {'NOVEL_ALL': 3.0, 'NOVEL_IRAE': 1.5, 'KNOWN': 1.0}
    PATENT_WATCH_MIN_REVERSAL = 20.0
    PATENT_WATCH_MIN_QUERIES  = 2

    MOA_REFERENCE = {
        'wz-3105':'SRC/ABL inhibitor', 'wz-4-145':'CDK8 inhibitor',
        'pf-431396':'FAK/PYK2 inhibitor', 'ql-xii-47':'MELK/FLT3 inhibitor',
        'ql-x-138':'BTK/MNK inhibitor', 'as-601245':'JNK inhibitor',
        'cgp-60474':'CDK1/2 inhibitor', 'azd-7762':'CHK1/2 inhibitor',
        'at-7519':'CDK1/2/4/6/9 inhibitor', 'alvocidib':'CDK1/2/4/6/9 inhibitor',
        'bi-2536':'PLK1 inhibitor', 'azd-5438':'CDK inhibitor',
        'canertinib':'Pan-EGFR inhibitor', 'gefitinib':'EGFR inhibitor',
        'nilotinib':'BCR-ABL inhibitor', 'dasatinib':'BCR-ABL/SRC inhibitor',
        'saracatinib':'SRC/ALK2 dual inhibitor', 'radicicol':'HSP90 inhibitor',
        'celastrol':'NF-kB/HSP90 inhibitor', 'withaferin-a':'NF-kB/HSP90 inhibitor',
        'nvp-auy922':'HSP90 inhibitor', 'pha-793887':'CDK inhibitor',
        'tofacitinib':'JAK1/3 inhibitor', 'baricitinib':'JAK1/2 inhibitor',
        'ruxolitinib':'JAK1/2 inhibitor', 'upadacitinib':'JAK1 inhibitor',
        'filgotinib':'JAK1 inhibitor', 'mitoxantrone':'Topoisomerase II inhibitor',
        'linifanib':'VEGFR/PDGFR inhibitor', 'fostamatinib':'SYK inhibitor',
        'rapamycin':'mTORC1 inhibitor', 'sirolimus':'mTORC1 inhibitor',
        'sb-431542':'TGF-beta receptor ALK5 inhibitor',
        'methotrexate':'Antifolate / immunosuppressant',
        'bms-387032':'unknown', 'hg-6-64-01':'unknown',
        'jw-7-24-1':'unknown', 'jnk-9l':'unknown',
    }

    def pubmed_hit_count(query, retries=3):
        params = {'db':'pubmed','term':query,'rettype':'count',
                  'retmode':'json','email':NCBI_EMAIL}
        for attempt in range(retries):
            try:
                resp = requests.get(NCBI_ESEARCH, params=params, timeout=10)
                resp.raise_for_status()
                count = int(resp.json()['esearchresult']['count'])
                time.sleep(NCBI_DELAY)
                return count
            except Exception:
                if attempt < retries - 1: time.sleep(NCBI_DELAY * 3)
                else: return -1

    def assess_novelty(compound_name):
        q = f'"{compound_name}"'
        hits_irae   = pubmed_hit_count(q + ' AND ("immune-related adverse" OR "irAE" OR "checkpoint arthritis")')
        hits_ici    = pubmed_hit_count(q + ' AND ("immune checkpoint" OR "PD-1" OR "CTLA-4" OR "anti-PD")')
        hits_arth   = pubmed_hit_count(q + ' AND ("rheumatoid arthritis" OR "inflammatory arthritis" OR "synovitis")')
        if hits_irae == 0 and hits_ici == 0 and hits_arth == 0: tier = 'NOVEL_ALL'
        elif hits_irae == 0 and hits_ici == 0:                   tier = 'NOVEL_IRAE'
        else:                                                     tier = 'KNOWN'
        return {'compound': compound_name, 'hits_irae': hits_irae,
                'hits_immune_checkpoint': hits_ici,
                'hits_arthritis': hits_arth, 'novelty_tier': tier}

    print('Loading LINCS candidates...')
    candidates = pd.read_csv(_s06_out)
    print(f'  {len(candidates)} candidates')

    novelty_rows = []
    for i, row in candidates.iterrows():
        compound = row['compound']
        print(f'  [{i+1}/{len(candidates)}] {compound}...', end=' ', flush=True)
        nov = assess_novelty(compound)
        novelty_rows.append(nov)
        print(f'{nov["novelty_tier"]} '
              f'(irAE:{nov["hits_irae"]}, ICI:{nov["hits_immune_checkpoint"]}, '
              f'Arth:{nov["hits_arthritis"]})')

    novelty_df = pd.DataFrame(novelty_rows)
    novelty_df.to_csv(os.path.join(PROCESSED, 'novelty_raw.csv'), index=False)

    merged = candidates.merge(novelty_df, on='compound', how='left')
    merged['novelty_tier']   = merged['novelty_tier'].fillna('KNOWN')
    merged['moa']            = merged['compound'].apply(
        lambda x: MOA_REFERENCE.get(x.lower().strip(), 'unknown'))
    merged['priority_score'] = merged.apply(
        lambda r: round(r['max_reversal_score'] *
                        NOVELTY_WEIGHTS.get(r['novelty_tier'], 1.0) *
                        r['n_queries'], 1), axis=1)
    merged = merged.sort_values('priority_score', ascending=False)

    print('\nNovelty breakdown:')
    print(merged['novelty_tier'].value_counts().to_string())

    display_cols = ['compound','moa','novelty_tier','max_reversal_score','n_queries','priority_score']
    print('\nTop 20 priority candidates:')
    print(merged[display_cols].head(20).round(2).to_string(index=False))

    patent_watch = merged[
        (merged['novelty_tier'] == 'NOVEL_ALL') &
        (merged['max_reversal_score'] >= PATENT_WATCH_MIN_REVERSAL) &
        (merged['n_queries'] >= PATENT_WATCH_MIN_QUERIES)
    ].copy()
    print(f'\nPatent watch: {len(patent_watch)} NOVEL_ALL compounds')
    if not patent_watch.empty:
        print(patent_watch[display_cols].to_string(index=False))

    merged.to_csv(_s07_out, index=False)
    patent_watch.to_csv(os.path.join(PROCESSED, 'patent_watch.csv'), index=False)
    print(f'\nSaved: priority_candidates.csv ({len(merged)} compounds)')
    print(f'Saved: patent_watch.csv ({len(patent_watch)} compounds)')
    print('\nScript 07 complete.')
    print('=' * 60)
    print('PIPELINE COMPLETE — review priority_candidates.csv')
    print('=' * 60)